# SSDA DANN Full Pipeline (Kaggle)

This notebook starts from dataset merge and covers: vectorization, merge, SSDA splits, DANN training, baselines, and Q-error/MAPE evaluation.

## 1) Load CSVs and Normalize Column Names
## 2) Create/Validate Domain-Adaptation Structure
## 3) Validate Keys and Domain Labels
## 4) Parse/Clean Query Plans
## 5) Vectorize Query Plans
## 6) Build Row-Level Plan Features
## 7) Select Internal Metrics + Knob Groups
## 8) Merge Run-History with Collected Features
## 9) Assemble Final Input Vector x
## 10) Construct SSDA Splits
## 11) Build DANN Modules
## 12) Residual Regression Head
## 13) Joint Training Loop
## 14) Baselines
## 15) Evaluation (MAPE, Q-Error)
## 16) Tune lambda/beta + Export Artifacts

In [ ]:
import os
import ast
import json
import random
import subprocess
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cpu')

In [2]:
# Kaggle paths example:
COLLECTED_CSV = '/kaggle/input/<dataset>/cost_model_collected.csv'
DOMAIN_CSV = '/kaggle/input/<dataset>/cost_model_run_history_domain_adaptation.csv'
STRUCTURED_CSV = '/kaggle/input/<dataset>/cost_model_run_history_domain_adaptation_structured.csv'
WORK_DIR = '/kaggle/working'

# Local fallback (if running outside Kaggle)
# COLLECTED_CSV = 'surrogate/cost_model_collected.csv'
# DOMAIN_CSV = 'surrogate/cost_model_run_history_domain_adaptation.csv'
# STRUCTURED_CSV = 'surrogate/cost_model_run_history_domain_adaptation_structured.csv'
# WORK_DIR = 'surrogate'

MERGE_SCRIPT = 'surrogate/merge_domain_adaptation_with_collected.py'

# 5+6: optional embedding generation path if qp_emb_vector is missing
USE_MINILM_IF_MISSING = False

# If pre-merged structured CSV exists, use it directly (no merge script needed).
if os.path.exists(STRUCTURED_CSV):
    print('Using pre-merged structured CSV:', STRUCTURED_CSV)
    df = pd.read_csv(STRUCTURED_CSV, low_memory=False)
else:
    # Fallback path: build qp_emb_vector (optional) + merge using script
    col_df = pd.read_csv(COLLECTED_CSV, low_memory=False)
    if 'qp_emb_vector' not in col_df.columns and USE_MINILM_IF_MISSING:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

        def parse_plans(v):
            if pd.isna(v):
                return []
            txt = str(v).strip()
            if not txt:
                return []
            for p in (json.loads, ast.literal_eval):
                try:
                    x = p(txt)
                    if isinstance(x, list):
                        return [str(i) for i in x]
                except Exception:
                    pass
            return [txt]

        texts = [' [SEP] '.join(parse_plans(v))[:4000] for v in col_df['collected.query_plans']]
        emb = model.encode(texts, batch_size=128, convert_to_numpy=True, normalize_embeddings=True)
        col_df['qp_emb_vector'] = [json.dumps(row.tolist()) for row in emb]
        col_df.to_csv(COLLECTED_CSV, index=False)

    if not os.path.exists(MERGE_SCRIPT):
        raise FileNotFoundError(
            f"{STRUCTURED_CSV} not found and merge script is missing: {MERGE_SCRIPT}"
        )

    cmd = [
        'python', MERGE_SCRIPT,
        '--domain-csv', DOMAIN_CSV,
        '--collected-csv', COLLECTED_CSV,
        '--output-csv', STRUCTURED_CSV,
    ]
    subprocess.run(cmd, check=True)
    df = pd.read_csv(STRUCTURED_CSV, low_memory=False)

print('rows:', len(df), 'cols:', len(df.columns))
print(df['domain_label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/<dataset>/cost_model_collected.csv'

In [ ]:
# 7+9: feature groups
meta_cols = [
    'target.cost', 'domain_label',
    'metadata.benchmark', 'metadata.db_engine', 'metadata.hardware',
    'metadata.hardware_specs.cores', 'metadata.hardware_specs.ram_gb', 'metadata.hardware_specs.threads',
    'metadata.source_run_history', 'metadata.workload', 'metadata.workload_key',
]

required = ['metadata.workload_key', 'domain_label', 'target.cost']
for c in required:
    assert c in df.columns, f'Missing required column: {c}'

assert set(df['domain_label'].dropna().unique()).issubset({0, 1}), 'domain_label must be 0/1'

feature_cols = [c for c in df.columns if c not in meta_cols]
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

df['target.cost'] = pd.to_numeric(df['target.cost'], errors='coerce')
df = df.dropna(subset=['target.cost']).reset_index(drop=True)

X = df[feature_cols].values.astype(np.float32)
y_raw = np.clip(df['target.cost'].values.astype(np.float32), a_min=0.0, a_max=None)
y = np.log1p(y_raw).astype(np.float32)
d = df['domain_label'].astype(int).values

print('X shape:', X.shape)
print('target source:', int((d == 0).sum()), 'target mysql:', int((d == 1).sum()))
print('y_raw min/max:', float(np.min(y_raw)), float(np.max(y_raw)))
print('y_log min/max:', float(np.min(y)), float(np.max(y)))
print('X finite:', bool(np.isfinite(X).all()), 'y finite:', bool(np.isfinite(y).all()))

# save feature index map
with open(os.path.join(WORK_DIR, 'feature_index_map.json'), 'w') as f:
    json.dump({k: i for i, k in enumerate(feature_cols)}, f, indent=2)

In [ ]:
# 10: SSDA splits
source_idx = np.where(d == 0)[0]
target_idx = np.where(d == 1)[0]

target_dev, target_test = train_test_split(target_idx, test_size=0.2, random_state=SEED)
target_tu, target_tl = train_test_split(target_dev, test_size=0.15, random_state=SEED)

print('D_S', len(source_idx))
print('D_TU', len(target_tu))
print('D_TL', len(target_tl))
print('D_TEST', len(target_test))

fit_idx = np.concatenate([source_idx, target_tu, target_tl])
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[fit_idx] = scaler.fit_transform(X[fit_idx])
X_scaled[target_test] = scaler.transform(X[target_test])

def make_loader(idx, batch=256, with_label=True, shuffle=True):
    xb = torch.tensor(X_scaled[idx], dtype=torch.float32)
    db = torch.tensor(d[idx], dtype=torch.float32).unsqueeze(1)
    if with_label:
        yb = torch.tensor(y[idx], dtype=torch.float32).unsqueeze(1)
        ds = TensorDataset(xb, yb, db)
    else:
        ds = TensorDataset(xb, db)
    # drop_last=False keeps all data, especially important for small D_TL
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, drop_last=False)

loader_s = make_loader(source_idx, 256, True)
loader_tu = make_loader(target_tu, 128, False)
loader_tl = make_loader(target_tl, 32, True)

print('loader sizes:', len(loader_s), len(loader_tu), len(loader_tl))

In [ ]:
# 11+12: DANN modules and residual regression head
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_out):
        return -ctx.lam * grad_out, None

def grad_reverse(x, lam):
    return GRL.apply(x, lam)

class Gf(nn.Module):
    def __init__(self, in_dim, z=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, z), nn.ReLU()
)
    def forward(self, x):
        return self.net(x)

class Gd(nn.Module):
    def __init__(self, z=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, z):
        return self.net(z)

class GyResidual(nn.Module):
    def __init__(self, z=256):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(z, 128), nn.ReLU(), nn.Linear(128, 1))
        self.delta = nn.Sequential(nn.Linear(z, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, z, dom):
        b = self.base(z)
        dlt = self.delta(z)
        yhat = b + dom * dlt
        return yhat, b, dlt

gf = Gf(X_scaled.shape[1], 256).to(DEVICE)
gd = Gd(256).to(DEVICE)
gy = GyResidual(256).to(DEVICE)

task_loss_fn = nn.SmoothL1Loss()
domain_loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(
    list(gf.parameters()) + list(gd.parameters()) + list(gy.parameters()),
    lr=2e-4,
    weight_decay=1e-5,
 )

In [ ]:
# 13: joint training loop
def lam_schedule(ep, max_ep):
    p = ep / max_ep
    lam = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0
    return float(min(lam, 0.7))  # cap early adversarial pressure for stability

EPOCHS = 40
BETA = 0.2

for ep in range(1, EPOCHS + 1):
    gf.train(); gd.train(); gy.train()
    lam = lam_schedule(ep, EPOCHS)

    # Use many optimization steps per epoch; cycle smaller loaders.
    steps = max(len(loader_s), len(loader_tu), len(loader_tl))
    it_s = cycle(loader_s)
    it_tu = cycle(loader_tu)
    it_tl = cycle(loader_tl)

    tot_task, tot_dom = 0.0, 0.0
    valid_steps = 0

    for _ in range(steps):
        xs, ys, ds = next(it_s)
        xtu, dtu = next(it_tu)
        xtl, ytl, dtl = next(it_tl)

        xs, ys, ds = xs.to(DEVICE), ys.to(DEVICE), ds.to(DEVICE)
        xtu, dtu = xtu.to(DEVICE), dtu.to(DEVICE)
        xtl, ytl, dtl = xtl.to(DEVICE), ytl.to(DEVICE), dtl.to(DEVICE)

        opt.zero_grad()

        zs = gf(xs)
        yhs, _, _ = gy(zs, ds)
        ls_task = task_loss_fn(yhs, ys)
        ls_dom_s = domain_loss_fn(gd(grad_reverse(zs, lam)), ds)

        ztu = gf(xtu)
        ls_dom_tu = domain_loss_fn(gd(grad_reverse(ztu, lam)), dtu)

        ztl = gf(xtl)
        yhtl, _, _ = gy(ztl, dtl)
        ltl_task = task_loss_fn(yhtl, ytl)
        ls_dom_tl = domain_loss_fn(gd(grad_reverse(ztl, lam)), dtl)

        task_loss = ls_task + ltl_task
        dom_loss = ls_dom_s + ls_dom_tu + ls_dom_tl
        loss = task_loss + BETA * dom_loss

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(gf.parameters()) + list(gd.parameters()) + list(gy.parameters()),
            max_norm=1.0,
        )
        opt.step()

        tot_task += float(task_loss.item())
        tot_dom += float(dom_loss.item())
        valid_steps += 1

    if ep % 5 == 0 or ep == 1:
        denom = max(valid_steps, 1)
        print(f'Epoch {ep:03d} | lam={lam:.3f} | task={tot_task/denom:.4f} | domain={tot_dom/denom:.4f} | valid_steps={valid_steps}/{steps}')

In [ ]:
# 14+15+16: baselines, evaluation, artifact export
@torch.no_grad()
def pred_dann(x, dom):
    gf.eval(); gy.eval()
    xb = torch.tensor(x, dtype=torch.float32, device=DEVICE)
    db = torch.tensor(dom.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    z = gf(xb)
    yhat, _, _ = gy(z, db)
    return yhat.squeeze(1).cpu().numpy()

def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100)

def qerr(y_true, y_pred, eps=1e-8):
    t = np.maximum(np.asarray(y_true), eps)
    p = np.maximum(np.asarray(y_pred), eps)
    q = np.maximum(p / t, t / p)
    return float(np.mean(q)), float(np.median(q)), float(np.quantile(q, 0.95))

class BaseMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, 1))
    def forward(self, x):
        return self.net(x)

def train_base(idx, epochs=35, bs=256):
    m = BaseMLP(X_scaled.shape[1]).to(DEVICE)
    o = torch.optim.Adam(m.parameters(), lr=2e-4)
    lfn = nn.SmoothL1Loss()
    dl = DataLoader(
        TensorDataset(
            torch.tensor(X_scaled[idx], dtype=torch.float32),
            torch.tensor(y[idx], dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=bs, shuffle=True, drop_last=True
    )
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            o.zero_grad()
            pr = m(xb)
            ls = lfn(pr, yb)
            if not torch.isfinite(ls):
                continue
            ls.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            o.step()
    return m

@torch.no_grad()
def pred_base(m, idx):
    m.eval()
    xb = torch.tensor(X_scaled[idx], dtype=torch.float32, device=DEVICE)
    return m(xb).squeeze(1).cpu().numpy()

# y is log1p(target), convert back for reporting
y_test = y_raw[target_test]
d_test = d[target_test]
x_test = X_scaled[target_test]

# DANN
pred_ssda_log = pred_dann(x_test, d_test)
pred_ssda = np.expm1(pred_ssda_log)

# zero-shot
m_zero = train_base(source_idx, epochs=35)
pred_zero_log = pred_base(m_zero, target_test)
pred_zero = np.expm1(pred_zero_log)

# few-shot
m_few = train_base(target_tl, epochs=60, bs=64)
pred_few_log = pred_base(m_few, target_test)
pred_few = np.expm1(pred_few_log)

rows = []
for name, pred in [('Zero-Shot', pred_zero), ('Few-Shot', pred_few), ('SSDA-DANN', pred_ssda)]:
    qm, qmed, q95 = qerr(y_test, pred)
    rows.append({'model': name, 'MAPE': mape(y_test, pred), 'QError_mean': qm, 'QError_median': qmed, 'QError_p95': q95})

res = pd.DataFrame(rows).sort_values('QError_mean')
res

# Export artifacts
torch.save({'gf': gf.state_dict(), 'gd': gd.state_dict(), 'gy': gy.state_dict()}, os.path.join(WORK_DIR, 'dann_checkpoint.pt'))
with open(os.path.join(WORK_DIR, 'split_summary.json'), 'w') as f:
    json.dump({'D_S': int(len(source_idx)), 'D_TU': int(len(target_tu)), 'D_TL': int(len(target_tl)), 'D_TEST': int(len(target_test))}, f, indent=2)
res.to_csv(os.path.join(WORK_DIR, 'evaluation_summary.csv'), index=False)
print('Saved artifacts in', WORK_DIR)